<a href="https://colab.research.google.com/github/Yashika-Bayeen/agentic-ai/blob/main/Day_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.2 MB/s eta 0:00:00


In [ ]:
from fastapi.responses import StreamingResponse
from langchain_groq import ChatGroq

def get_agent():
    return ChatGroq(model="llama-3.3-70b-versatile",
                    api_key=os.environ["GROQ_API_KEY"], temperature=0)

async def token_stream(question: str):
    llm = get_agent()
    # 🌊 .stream() yields chunks as the model produces them
    for chunk in llm.stream(question):
        text = chunk.content
        if text:
            yield f"data: {text}\n\n"      # SSE format: 'data: ...\n\n'
    yield "data: [DONE]\n\n"               # a sentinel so the client knows we're finished

@app.post("/ask/stream")
async def ask_stream(req: AskRequest):
    return StreamingResponse(
        token_stream(req.question),
        media_type="text/event-stream",    # 🌊 the SSE content type
    )